In [1]:
import os
from datetime import datetime
from zoneinfo import ZoneInfo

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found. Check your .env file.")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=api_key,
    temperature=0
)

print("Groq model initialized successfully.")

Groq model initialized successfully.


In [2]:
calendar_data = {
    ("Aarav", "2026-10-26"): {
        "available": True,
        "busy_times": ["14:00", "15:00"]
    },
    ("Priya", "2026-10-26"): {
        "available": False,
        "busy_times": ["10:00", "11:00", "14:00"]
    }
}

booked_meetings = []

print("Sample calendar data created.")
print("Existing bookings:", booked_meetings)

Sample calendar data created.
Existing bookings: []


In [3]:
@tool
def check_calendar_availability(person: str, date: str) -> str:
    """Check whether a person is generally available on a given date using the sample calendar."""

    try:
        datetime.strptime(date, "%Y-%m-%d")
    except ValueError:
        return "Invalid date. Use YYYY-MM-DD format."

    key = (person.strip().title(), date)

    if key not in calendar_data:
        return (
            f"No calendar record found for {person} on {date}. "
            "Availability is unknown."
        )

    record = calendar_data[key]

    if record["available"]:
        return (
            f"{person} is generally available on {date}. "
            f"Busy times: {', '.join(record['busy_times']) or 'None'}."
        )

    return (
        f"{person} is marked unavailable on {date}. "
        f"Busy times: {', '.join(record['busy_times'])}."
    )


print(check_calendar_availability.invoke({
    "person": "Aarav",
    "date": "2026-10-26"
}))

Aarav is generally available on 2026-10-26. Busy times: 14:00, 15:00.


In [4]:
@tool
def get_timezone_offset(city: str) -> str:
    """Get the current UTC offset and timezone name for a supported city."""

    city_timezones = {
        "mumbai": "Asia/Kolkata",
        "delhi": "Asia/Kolkata",
        "pune": "Asia/Kolkata",
        "london": "Europe/London",
        "new york": "America/New_York",
        "tokyo": "Asia/Tokyo"
    }

    city_key = city.strip().lower()
    timezone_name = city_timezones.get(city_key)

    if timezone_name is None:
        return (
            f"Timezone not configured for {city}. "
            "Please add the city's IANA timezone."
        )

    now = datetime.now(ZoneInfo(timezone_name))
    offset = now.strftime("%z")
    formatted_offset = f"UTC{offset[:3]}:{offset[3:]}"

    return (
        f"City: {city.title()}\n"
        f"Timezone: {timezone_name}\n"
        f"Current UTC offset: {formatted_offset}"
    )


print(get_timezone_offset.invoke({"city": "Mumbai"}))

City: Mumbai
Timezone: Asia/Kolkata
Current UTC offset: UTC+05:30


In [5]:
@tool
def book_meeting(person: str, date: str, time: str) -> str:
    """Book a meeting in the sample calendar for a person, date, and 24-hour time."""

    try:
        meeting_date = datetime.strptime(date, "%Y-%m-%d").date()
        datetime.strptime(time, "%H:%M")
    except ValueError:
        return "Invalid date or time. Use YYYY-MM-DD and HH:MM formats."

    if meeting_date < datetime.now().date():
        return "Booking rejected: the meeting date is in the past."

    person_name = person.strip().title()
    key = (person_name, date)

    if key not in calendar_data:
        return (
            f"Booking rejected: no calendar record for {person_name} "
            f"on {date}; availability is unknown."
        )

    record = calendar_data[key]

    if not record["available"]:
        return f"Booking rejected: {person_name} is marked unavailable on {date}."

    if time in record["busy_times"]:
        return f"Booking rejected: {person_name} is busy at {time} on {date}."

    if any(
        booking["person"] == person_name
        and booking["date"] == date
        and booking["time"] == time
        for booking in booked_meetings
    ):
        return "Booking rejected: this meeting is already booked."

    booking = {
        "person": person_name,
        "date": date,
        "time": time,
        "status": "Confirmed"
    }

    booked_meetings.append(booking)

    return (
        f"Meeting booked successfully.\n"
        f"Person: {person_name}\n"
        f"Date: {date}\n"
        f"Time: {time}\n"
        f"Status: Confirmed"
    )


print("Meeting booking tool created.")

Meeting booking tool created.


In [6]:
tools = [
    check_calendar_availability,
    get_timezone_offset,
    book_meeting
]

tool_map = {
    item.name: item for item in tools
}

llm_with_tools = llm.bind_tools(tools)

print("Tools bound to the model:")
for item in tools:
    print("-", item.name)

Tools bound to the model:
- check_calendar_availability
- get_timezone_offset
- book_meeting


In [7]:
availability_query = (
    "Is Aarav available on 2026-10-26? "
    "Only check the calendar availability. Do not book a meeting "
    "or look up a timezone."
)

availability_messages = [
    HumanMessage(content=availability_query)
]

availability_response = llm_with_tools.invoke(availability_messages)

print("Query 1:", availability_query)
print("\nModel-suggested tool calls:")

for call in availability_response.tool_calls:
    print("Tool:", call["name"])
    print("Arguments:", call["args"])

if not availability_response.tool_calls:
    raise RuntimeError("No tool call suggested for the availability query.")

Query 1: Is Aarav available on 2026-10-26? Only check the calendar availability. Do not book a meeting or look up a timezone.

Model-suggested tool calls:
Tool: check_calendar_availability
Arguments: {'date': '2026-10-26', 'person': 'Aarav'}


In [8]:
availability_messages.append(availability_response)

availability_results = []

for call in availability_response.tool_calls:
    if call["name"] != "check_calendar_availability":
        raise ValueError(
            f"Unexpected tool for availability-only query: {call['name']}"
        )

    result = tool_map[call["name"]].invoke(call["args"])
    availability_results.append(result)

    availability_messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=call["id"]
        )
    )

availability_final = llm.invoke([
    HumanMessage(
        content=(
            "Summarize this calendar lookup result without booking "
            "a meeting or calling any more tools:\n\n"
            + "\n\n".join(availability_results)
        )
    )
])

print("Availability result:")
print("\n".join(availability_results))
print("\nFinal answer:")
print(availability_final.content)

Availability result:
Aarav is generally available on 2026-10-26. Busy times: 14:00, 15:00.

Final answer:
Aarav is free on 2026‑10‑26, with only two conflicts that day: he’s busy at 14:00 – 15:00 and again at 15:00 – 16:00. All other times are open.


In [9]:
booking_query = """
Arrange a meeting with Aarav on 2026-10-26 at 10:00 Mumbai time.

Use these tools in sequence:
1. check_calendar_availability for Aarav on 2026-10-26.
2. get_timezone_offset for Mumbai.
3. book_meeting for Aarav on 2026-10-26 at 10:00.

Do not book if availability is unknown or the person is unavailable.
"""

booking_messages = [HumanMessage(content=booking_query)]

booking_response = llm_with_tools.invoke(booking_messages)

print("Query 2:", booking_query)
print("\nFirst-round model-suggested tool calls:")

for call in booking_response.tool_calls:
    print("Tool:", call["name"])
    print("Arguments:", call["args"])

Query 2: 
Arrange a meeting with Aarav on 2026-10-26 at 10:00 Mumbai time.

Use these tools in sequence:
1. check_calendar_availability for Aarav on 2026-10-26.
2. get_timezone_offset for Mumbai.
3. book_meeting for Aarav on 2026-10-26 at 10:00.

Do not book if availability is unknown or the person is unavailable.


First-round model-suggested tool calls:
Tool: check_calendar_availability
Arguments: {'date': '2026-10-26', 'person': 'Aarav'}


In [10]:
booking_messages.append(booking_response)

booking_results = []
completed_tools = set()
availability_text = ""
timezone_text = ""

def execute_suggested_call(call):
    tool_name = call["name"]

    if tool_name not in tool_map:
        raise ValueError(f"Unknown tool: {tool_name}")

    result = tool_map[tool_name].invoke(call["args"])
    booking_results.append((tool_name, result))
    completed_tools.add(tool_name)

    if tool_name == "check_calendar_availability":
        global_availability = result
        return result

    return result


# Execute calls suggested in the first round
for call in booking_response.tool_calls:
    result = execute_suggested_call(call)

    if call["name"] == "check_calendar_availability":
        availability_text = result
    elif call["name"] == "get_timezone_offset":
        timezone_text = result

    booking_messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=call["id"]
        )
    )

# Run missing prerequisites if the model omitted them
if "check_calendar_availability" not in completed_tools:
    availability_text = check_calendar_availability.invoke({
        "person": "Aarav",
        "date": "2026-10-26"
    })
    booking_results.append((
        "check_calendar_availability",
        availability_text
    ))
    completed_tools.add("check_calendar_availability")

if "get_timezone_offset" not in completed_tools:
    timezone_text = get_timezone_offset.invoke({"city": "Mumbai"})
    booking_results.append(("get_timezone_offset", timezone_text))
    completed_tools.add("get_timezone_offset")

print("Availability result:")
print(availability_text)

print("\nTimezone result:")
print(timezone_text)

if "marked unavailable" in availability_text or "No calendar record" in availability_text:
    print("\nBooking skipped because availability was not confirmed.")
else:
    booking_request = HumanMessage(
        content=(
            "Based on the actual results below, suggest a call to "
            "book_meeting for Aarav on 2026-10-26 at 10:00 only if "
            "the calendar confirms availability. Do not invent results.\n\n"
            f"Availability:\n{availability_text}\n\n"
            f"Timezone:\n{timezone_text}"
        )
    )

    booking_suggestion = llm_with_tools.invoke([booking_request])

    print("\nBooking-round model-suggested tool calls:")
    for call in booking_suggestion.tool_calls:
        print("Tool:", call["name"])
        print("Arguments:", call["args"])

    booking_call_found = False

    for call in booking_suggestion.tool_calls:
        if call["name"] == "book_meeting":
            booking_call_found = True
            booking_result = book_meeting.invoke(call["args"])
            booking_results.append(("book_meeting", booking_result))
            completed_tools.add("book_meeting")

            print("\nBooking result:")
            print(booking_result)

    if not booking_call_found:
        print("\nThe model did not suggest book_meeting.")
        print("No booking was made.")

Availability result:
Aarav is generally available on 2026-10-26. Busy times: 14:00, 15:00.

Timezone result:
City: Mumbai
Timezone: Asia/Kolkata
Current UTC offset: UTC+05:30

Booking-round model-suggested tool calls:
Tool: book_meeting
Arguments: {'date': '2026-10-26', 'person': 'Aarav', 'time': '10:00'}

Booking result:
Meeting booked successfully.
Person: Aarav
Date: 2026-10-26
Time: 10:00
Status: Confirmed


In [11]:
print("All results from Query 2:")

for name, result in booking_results:
    print(f"\n{name}:")
    print(result)

assert any(
    name == "check_calendar_availability"
    for name, result in booking_results
), "Availability was not checked."

assert any(
    name == "get_timezone_offset"
    for name, result in booking_results
), "Timezone was not retrieved."

if any(
    name == "book_meeting" and "Meeting booked successfully" in result
    for name, result in booking_results
):
    print("\nBooking workflow completed successfully.")
else:
    print("\nBooking was not confirmed; inspect the results above.")

print("\nQuery 1 tool calls:")
print([call["name"] for call in availability_response.tool_calls])

print("\nQuery 2 initial tool calls:")
print([call["name"] for call in booking_response.tool_calls])

All results from Query 2:

check_calendar_availability:
Aarav is generally available on 2026-10-26. Busy times: 14:00, 15:00.

get_timezone_offset:
City: Mumbai
Timezone: Asia/Kolkata
Current UTC offset: UTC+05:30

book_meeting:
Meeting booked successfully.
Person: Aarav
Date: 2026-10-26
Time: 10:00
Status: Confirmed

Booking workflow completed successfully.

Query 1 tool calls:
['check_calendar_availability']

Query 2 initial tool calls:
['check_calendar_availability']


In [12]:
print("Lab 6 - Exercise 9 completed.")
print("Three @tool functions were created and bound together.")
print("Query 1 tested availability-only tool selection.")
print("Query 2 tested availability, timezone lookup, and booking.")
print("All records use a simulated calendar, not a real calendar service.")

Lab 6 - Exercise 9 completed.
Three @tool functions were created and bound together.
Query 1 tested availability-only tool selection.
Query 2 tested availability, timezone lookup, and booking.
All records use a simulated calendar, not a real calendar service.
